# Real-World LightGBM Demand Forecasting
**Best possible ML approach** — treating this as genuine retail data with noise, seasonality, and trends.

## 1. Combine Train + Test for Feature Engineering

In [ ]:
# Tag train vs test
train_raw['is_train'] = 1
test_raw['is_train'] = 0
if 'sales' not in test_raw.columns:
    test_raw['sales'] = np.nan

df = pd.concat([train_raw, test_raw], ignore_index=True)
df = df.sort_values(['store', 'item', 'date']).reset_index(drop=True)
print(f'Combined: {df.shape}')


## 2. Date Features

In [ ]:
df['year'] = df.date.dt.year
df['month'] = df.date.dt.month
df['day'] = df.date.dt.day
df['dayofweek'] = df.date.dt.dayofweek
df['dayofyear'] = df.date.dt.dayofyear
df['weekofyear'] = df.date.dt.isocalendar().week.astype(int)
df['is_weekend'] = (df.dayofweek >= 5).astype(int)
df['is_month_start'] = df.date.dt.is_month_start.astype(int)
df['is_month_end'] = df.date.dt.is_month_end.astype(int)
df['quarter'] = df.date.dt.quarter

# Days since start (continuous trend feature)
df['days_since_start'] = (df.date - df.date.min()).dt.days

print('Date features done.')
print(df[['date','year','month','dayofweek','is_weekend','quarter','days_since_start']].head(10))


## 3. Lag Features
We use lags of 91+ days (since we must forecast 90 days ahead, shorter lags would leak).

In [ ]:
# Lag features — only lags >= 91 days are valid (no data leakage for 90-day forecast)
lag_days = [91, 98, 105, 112, 182, 364, 365, 728]

for lag in lag_days:
    col_name = f'sales_lag_{lag}'
    df[col_name] = df.groupby(['store', 'item'])['sales'].shift(lag)
    print(f'  Created {col_name}')

print(f'Lag features done. Shape: {df.shape}')


## 4. Rolling Window Features
Rolling mean/std over windows anchored at lag=91 (safe from leakage).

In [ ]:
# Rolling statistics computed on the shifted (lag-91) series
windows = [7, 14, 28, 56, 91]

for w in windows:
    # Rolling mean
    col_mean = f'rolling_mean_{w}'
    df[col_mean] = df.groupby(['store', 'item'])['sales'].transform(
        lambda x: x.shift(91).rolling(window=w, min_periods=1).mean()
    )
    # Rolling std
    col_std = f'rolling_std_{w}'
    df[col_std] = df.groupby(['store', 'item'])['sales'].transform(
        lambda x: x.shift(91).rolling(window=w, min_periods=1).std()
    )
    print(f'  Created {col_mean}, {col_std}')

print(f'Rolling features done. Shape: {df.shape}')


## 5. Expanding Window Features

In [ ]:
# Expanding mean (all history up to lag-91)
df['expanding_mean'] = df.groupby(['store', 'item'])['sales'].transform(
    lambda x: x.shift(91).expanding(min_periods=1).mean()
)
print('Expanding mean done.')


## 6. Target Encoding Features
Mean sales per store, per item, per store-item, per month, per dayofweek — computed on TRAINING data only.

In [ ]:
train_mask = df['is_train'] == 1

# Store-level mean
store_mean = df.loc[train_mask].groupby('store')['sales'].mean()
df['store_mean'] = df['store'].map(store_mean)

# Item-level mean
item_mean = df.loc[train_mask].groupby('item')['sales'].mean()
df['item_mean'] = df['item'].map(item_mean)

# Store-Item mean
store_item_mean = df.loc[train_mask].groupby(['store', 'item'])['sales'].mean()
df['store_item_mean'] = df.set_index(['store', 'item']).index.map(store_item_mean)

# Month mean
month_mean = df.loc[train_mask].groupby('month')['sales'].mean()
df['month_mean'] = df['month'].map(month_mean)

# DayOfWeek mean
dow_mean = df.loc[train_mask].groupby('dayofweek')['sales'].mean()
df['dow_mean'] = df['dayofweek'].map(dow_mean)

# Store-Month interaction
store_month_mean = df.loc[train_mask].groupby(['store', 'month'])['sales'].mean()
df['store_month_mean'] = df.set_index(['store', 'month']).index.map(store_month_mean)

# Item-Month interaction
item_month_mean = df.loc[train_mask].groupby(['item', 'month'])['sales'].mean()
df['item_month_mean'] = df.set_index(['item', 'month']).index.map(item_month_mean)

# Item-DayOfWeek interaction
item_dow_mean = df.loc[train_mask].groupby(['item', 'dayofweek'])['sales'].mean()
df['item_dow_mean'] = df.set_index(['item', 'dayofweek']).index.map(item_dow_mean)

# Store-DayOfWeek interaction
store_dow_mean = df.loc[train_mask].groupby(['store', 'dayofweek'])['sales'].mean()
df['store_dow_mean'] = df.set_index(['store', 'dayofweek']).index.map(store_dow_mean)

print('Target encoding features done.')


## 7. Year-over-Year Growth Features

In [ ]:
# YoY ratio: sales_lag_364 / sales_lag_728 — captures growth trend
df['yoy_ratio'] = df['sales_lag_364'] / df['sales_lag_728'].replace(0, np.nan)

# Same-month last year average
df['sales_same_month_last_year'] = df.groupby(['store', 'item', 'month'])['sales'].transform(
    lambda x: x.shift(12)  # shift by 12 months (approximate with groupby month)
)

print('YoY features done.')


## 8. Prepare Train/Validation/Test Splits

In [ ]:
# Define feature columns (exclude non-feature columns)
exclude_cols = ['date', 'sales', 'is_train', 'id']
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f'Number of features: {len(feature_cols)}')
print(f'Features: {feature_cols}')

# Training data: 2014-2017 (drop 2013 — not enough lag history)
train_df = df[(df['is_train'] == 1) & (df['year'] >= 2014)].copy()
test_df = df[df['is_train'] == 0].copy()

# Validation: last 3 months of training (Oct-Dec 2017) to match test seasonality (Jan-Mar 2018)
# Actually, let's use Jan-Mar 2017 to better match the test months
val_mask = (train_df['date'] >= '2017-01-01') & (train_df['date'] <= '2017-03-31')
val_df = train_df[val_mask]
fit_df = train_df[~val_mask]

print(f'Fit set:  {fit_df.shape[0]:,} rows ({fit_df.date.min()} to {fit_df.date.max()})')
print(f'Val set:  {val_df.shape[0]:,} rows ({val_df.date.min()} to {val_df.date.max()})')
print(f'Test set: {test_df.shape[0]:,} rows ({test_df.date.min()} to {test_df.date.max()})')

# Drop rows with NaN in target
fit_df = fit_df.dropna(subset=['sales'])
val_df = val_df.dropna(subset=['sales'])


## 9. Train LightGBM

In [ ]:
# SMAPE metric for LightGBM evaluation
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 200.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.nanmean(diff)

def lgb_smape(preds, train_data):
    labels = train_data.get_label()
    return 'smape', smape(labels, preds), False

# Create LightGBM datasets
X_fit = fit_df[feature_cols]
y_fit = fit_df['sales']
X_val = val_df[feature_cols]
y_val = val_df['sales']

lgb_train = lgb.Dataset(X_fit, y_fit)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

# Hyperparameters (well-tuned for this type of data)
params = {
    'objective': 'regression_l1',  # MAE — closer to SMAPE optimization than MSE
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'learning_rate': 0.02,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'max_depth': -1,
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42,
}

print('Training LightGBM...')
t0 = time.time()

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    feval=lgb_smape,
    callbacks=[
        lgb.early_stopping(200),
        lgb.log_evaluation(500),
    ],
)

print(f'Training done in {time.time()-t0:.1f}s')
print(f'Best iteration: {model.best_iteration}')


## 10. Feature Importance

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print('Top 20 features by gain:')
print(importance.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(25).plot.barh(x='feature', y='importance', ax=ax)
ax.set_title('Top 25 Feature Importances (Gain)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()


## 11. Validation Score

In [ ]:
val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_preds = np.clip(val_preds, 0, None)  # sales can't be negative
val_smape = smape(y_val.values, val_preds)
print(f'Validation SMAPE: {val_smape:.5f}')


## 12. Also Train on FULL Training Data for Final Predictions

In [ ]:
# Retrain on ALL training data (2014-2017) using the best iteration count
X_full = train_df[feature_cols].copy()
y_full = train_df['sales'].copy()

# Drop NaN rows
valid_mask = y_full.notna()
X_full = X_full[valid_mask]
y_full = y_full[valid_mask]

lgb_full = lgb.Dataset(X_full, y_full)

print(f'Retraining on full data ({X_full.shape[0]:,} rows) for {model.best_iteration} rounds...')
t0 = time.time()

model_full = lgb.train(
    params,
    lgb_full,
    num_boost_round=model.best_iteration,
    valid_sets=[lgb_full],
    valid_names=['full'],
    callbacks=[lgb.log_evaluation(500)],
)

print(f'Full training done in {time.time()-t0:.1f}s')


## 13. Generate Predictions

In [ ]:
X_test = test_df[feature_cols]
test_preds = model_full.predict(X_test, num_iteration=model_full.best_iteration)
test_preds = np.clip(test_preds, 0, None)

# CRITICAL: test_df was reordered by (store, item, date) during feature engineering.
# We must match predictions back to the original test IDs.
submission = test_df[['id']].copy()
submission['id'] = submission['id'].astype(int)
submission['sales'] = test_preds
submission = submission.sort_values('id').reset_index(drop=True)

submission.to_csv('../outputs/predictions.csv', index=False)
print('Saved ../outputs/predictions.csv')
print(submission.describe())
